In [ ]:
from pathlib import Path
import hashlib
import json

EXPECTED_REVISION = "be5050ba482607f90e607e044f03a2ebc38f98c3"
EXPECTED_WEIGHTS_SHA256 = (
    "d658fa8aad744c66d1abf43c2bf6bf594e75e834186dce9aadd1b7a9e4381bf0"
)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while chunk := file.read(chunk_size):
            digest.update(chunk)

    return digest.hexdigest()

candidates = [
    path.parent
    for path in Path("/kaggle/input").rglob("model.safetensors")
    if (path.parent / "config.json").exists()
    and (path.parent / "MODEL_PROVENANCE.json").exists()
]

if len(candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one Pillar-0 bundle, found {len(candidates)}: "
        f"{candidates}"
    )

MODEL_DIR = candidates[0]

provenance = json.loads(
    (MODEL_DIR / "MODEL_PROVENANCE.json").read_text()
)

weights_hash = sha256(MODEL_DIR / "model.safetensors")

assert provenance["source_revision"] == EXPECTED_REVISION
assert weights_hash == EXPECTED_WEIGHTS_SHA256
assert (MODEL_DIR / "LICENSE").exists()
assert (MODEL_DIR / "inventory.csv").exists()

print("Offline model directory:", MODEL_DIR)
print("Source repository:", provenance["source_repository"])
print("Source revision:", provenance["source_revision"])
print("License:", provenance["license"])
print("Weights SHA256:", weights_hash)
print("Bundle verification passed.")

In [ ]:
import gc
import os
import shutil
import sys
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
from safetensors.torch import load_file

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a Kaggle GPU, restart the session, and rerun the cells."
    )

PACKAGE_PARENT = Path("/kaggle/working")
PACKAGE_DIR = PACKAGE_PARENT / "pillar0_local"

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir()
(PACKAGE_DIR / "__init__.py").write_text("", encoding="utf-8")

for source_path in MODEL_DIR.glob("*.py"):
    shutil.copy2(
        source_path,
        PACKAGE_DIR / source_path.name,
    )

if str(PACKAGE_PARENT) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PARENT))

from pillar0_local.config_clip_mmatlas import (
    CLIPMultimodalAtlasConfig,
)
from pillar0_local.modeling_clip_mmatlas import (
    CLIPMultimodalAtlas,
)

config = CLIPMultimodalAtlasConfig.from_pretrained(
    MODEL_DIR,
    local_files_only=True,
)

# Construct the model on a real CPU device.
with torch.device("cpu"):
    pillar = CLIPMultimodalAtlas(config)

state_dict = load_file(
    str(MODEL_DIR / "model.safetensors"),
    device="cpu",
)

# Bypass Pillar-0's load_state_dict override. The saved keys correctly
# include the wrapper's "model." prefix.
load_result = torch.nn.Module.load_state_dict(
    pillar,
    state_dict,
    strict=True,
)

del state_dict
gc.collect()

device = torch.device("cuda:0")

pillar.eval()
pillar.requires_grad_(False)
pillar.to(device)

parameter_count = sum(
    parameter.numel()
    for parameter in pillar.parameters()
)

trainable_count = sum(
    parameter.numel()
    for parameter in pillar.parameters()
    if parameter.requires_grad
)

print("Model class:", type(pillar))
print("Parameters:", f"{parameter_count:,}")
print("Trainable parameters:", trainable_count)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Training mode:", pillar.training)
print("Device:", next(pillar.parameters()).device)
print("Parameter dtype:", next(pillar.parameters()).dtype)
print("Embedding dimension:", pillar.config.embed_dim)
print("Input image size:", pillar.config.vision_cfg["image_size"])
print("Offline Pillar-0 load passed.")

In [ ]:
import time

MODEL_INPUT_KEY = "breast_mr"
EXPECTED_OUTPUT_SHAPE = (1, 1152)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Synthetic input checks model execution without requiring competition data.
synthetic_volume = torch.zeros(
    (1, 3, 384, 384, 192),
    device=device,
    dtype=torch.float16,
)

torch.cuda.synchronize()
start_time = time.perf_counter()

with torch.inference_mode():
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        first_output = pillar.extract_vision_feats(
            {
                MODEL_INPUT_KEY: synthetic_volume,
            }
        )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start_time

if first_output.shape != EXPECTED_OUTPUT_SHAPE:
    raise RuntimeError(
        f"Unexpected output shape: {tuple(first_output.shape)}"
    )

if not torch.isfinite(first_output).all():
    raise RuntimeError("Output contains NaN or infinite values")

embedding = first_output.float().cpu()

print("Input shape:", tuple(synthetic_volume.shape))
print("Output shape:", tuple(first_output.shape))
print("Output finite:", bool(torch.isfinite(first_output).all()))
print("Embedding L2 norm:", float(torch.linalg.vector_norm(embedding)))
print("Inference time (seconds):", round(elapsed, 3))
print(
    "Peak GPU memory allocated (MB):",
    round(torch.cuda.max_memory_allocated() / 1024**2, 1),
)
print("Offline forward pass succeeded.")

del first_output
del embedding
del synthetic_volume

torch.cuda.empty_cache()